# xG model exploration

Reads everything from S3 through `nhl.storage.Store`. Run `nhl train-xg` first.

In [ ]:
import json
import matplotlib.pyplot as plt
import polars as pl

from nhl.storage import keys
from nhl.storage.s3 import Store
from nhl.models.xg import load_models

store = Store()
models, meta = load_models(store)
print(meta["version"], meta["split"])

## Test-set calibration by strength group

In [ ]:
fig, axes = plt.subplots(1, len(meta["groups"]), figsize=(16, 4))
for ax, (group, g) in zip(axes, meta["groups"].items()):
    cal = pl.DataFrame(g["test"]["calibration"]).filter(pl.col("n") > 0)
    ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
    ax.scatter(cal["mean_xg"], cal["goal_rate"], s=(cal["n"] / cal["n"].max() * 300).to_list())
    lim = max(cal["mean_xg"].max(), cal["goal_rate"].max()) * 1.1
    ax.set(xlim=(0, lim), ylim=(0, lim), title=f"{group}  AUC {g['test']['auc']:.3f}", xlabel="mean xG", ylabel="goal rate")
plt.tight_layout()

## Feature importance (gain)

In [ ]:
imp = (
    pl.DataFrame([{"group": g, "feature": f, "gain": v} for g, b in models.items() for f, v in b.get_score(importance_type="gain").items()])
    .with_columns((pl.col("gain") / pl.col("gain").sum().over("group")).alias("share"))
)
imp.filter(pl.col("group") == "EV").sort("share", descending=True).head(20)

## Shot map: mean xG by location (even strength, one season)

In [ ]:
season = 20242025
xg = store.get_parquet(keys.xg_predictions(season))
shots = store.get_parquet(keys.shots(season)).select("game_id", "event_idx", "x_abs", "y_abs")
ev = xg.filter(pl.col("strength_group") == "EV").join(shots, on=["game_id", "event_idx"])
grid = (
    ev.with_columns((pl.col("x_abs") // 4 * 4).alias("xb"), (pl.col("y_abs") // 4 * 4).alias("yb"))
    .group_by("xb", "yb").agg(pl.col("xg").mean(), pl.len()).filter(pl.col("len") >= 20)
)
plt.figure(figsize=(7, 6))
plt.scatter(grid["yb"], grid["xb"], c=grid["xg"], cmap="viridis", marker="s", s=60)
plt.colorbar(label="mean xG"); plt.xlabel("y (ft)"); plt.ylabel("x toward goal (ft)"); plt.ylim(0, 100)

## Team xG for / against per game (all situations)

In [ ]:
games = store.get_parquet(keys.GAMES).filter(pl.col("season") == season)
team = (
    xg.join(games.select("game_id", "home_abbr", "away_abbr"), on="game_id")
    .with_columns(pl.when(pl.col("event_team_abbr") == pl.col("home_abbr")).then(pl.col("away_abbr")).otherwise(pl.col("home_abbr")).alias("opponent"))
)
xgf = team.group_by("event_team_abbr").agg(pl.col("xg").sum().alias("xGF"), pl.col("is_goal").sum().alias("GF"), pl.col("game_id").n_unique().alias("gp"))
xga = team.group_by("opponent").agg(pl.col("xg").sum().alias("xGA"), pl.col("is_goal").sum().alias("GA"))
(
    xgf.join(xga, left_on="event_team_abbr", right_on="opponent")
    .with_columns((pl.col("xGF") / (pl.col("xGF") + pl.col("xGA"))).alias("xGF%"), (pl.col("GF") - pl.col("xGF")).alias("finishing"))
    .sort("xGF%", descending=True)
)